# PyTorch Training Pipeline using `nn.Module` & `torch.optim`
This notebook refactors the binary classification training pipeline. Instead of managing parameters, gradients, and optimization loops manually, we leverage PyTorch's native abstractions:
1. `nn.Module` to build the neural network layers.
2. `nn.BCELoss` (Binary Cross Entropy) for the classification objective function.
3. `torch.optim.SGD` (Stochastic Gradient Descent) for parameter updates and gradient clearing.

## 1. Importing Libraries
We import the neural network (`nn`) and optimization (`optim`) modules from PyTorch, alongside sklearn's preprocessing tools.

In [ ]:
# Importing core libraries
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

## 2. Loading the Dataset

In [ ]:
# Load Breast Cancer dataset
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

## 3. Data Inspection and Cleaning

In [ ]:
print(f"No of rows : {df.shape[0]} ")
print(f"No of columns : {df.shape[1]} ")

In [ ]:
# Drop non-informative features
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

In [ ]:
df.head()

In [ ]:
print(f"No of rows : {df.shape[0]} ")
print(f"No of columns : {df.shape[1]} ")

## 4. Preprocessing: Splitting and Scaling

In [ ]:
# Split dataset into training (80%) and test (20%) subsets
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2, random_state=42)

In [ ]:
# Apply feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 5. Preprocessing: Label Encoding

In [ ]:
y_train

In [ ]:
# Encode categorical diagnosis classes to 0 and 1
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [ ]:
y_train

## 6. Converting Data to PyTorch Tensors
We cast features and labels to Float32 tensors. This matches the default weight data type of PyTorch linear layers, avoiding type mismatch errors.

In [ ]:
# Convert to PyTorch float32 tensors (Fix: .float() avoids double-to-float layer type mismatches)
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).float()
y_test_tensor = torch.from_numpy(y_test).float()

In [ ]:
X_train_tensor.shape

In [ ]:
y_train_tensor.shape

## 7. Defining the Model class via `nn.Module`
We build a simple single-neuron neural network. It utilizes `nn.Linear` for linear mapping and `nn.Sigmoid` for producing probabilities.

In [ ]:
# Defining the model using PyTorch nn.Module
class MySimpleNN(nn.Module):
    def __init__(self, num_features): # Constructor
        super().__init__()
        # Linear layer mapping features to a single node output
        self.linear = nn.Linear(in_features=num_features, out_features=1)
        # Sigmoid activation function
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, features):
        # Sequential execution
        out = self.linear(features)
        out = self.sigmoid(out)
        return out

## 8. Training Parameters Configuration

In [ ]:
# Important Parameters
learning_rate = 0.1
epochs = 25

## 9. Defining the Loss Criterion
We use PyTorch's built-in `nn.BCELoss` (Binary Cross-Entropy Loss) wrapper.

In [ ]:
# Standard Binary Cross Entropy loss function
loss_function = nn.BCELoss()

## 10. Training Loop using Optimizer
We instantiate the model, configure the Stochastic Gradient Descent (SGD) optimizer, and run the training loop using the optimization pipeline.

In [ ]:
# Create the model instance
model = MySimpleNN(num_features=X_train_tensor.shape[1])

# Define optimizer (SGD), passing model parameter references and learning rate
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(epochs):
    # 1. Forward Pass
    y_pred = model(X_train_tensor)

    # 2. Loss calculation (Fix: Call global loss_function; view target as (N, 1) to avoid broadcasting)
    loss = loss_function(y_pred, y_train_tensor.view(-1, 1))

    # 3. Clear existing gradients (Optimizer zeroes the tracked parameter gradients)
    optimizer.zero_grad()

    # 4. Backward Pass
    loss.backward()

    # 5. Parameters update (Optimizer updates weight and bias parameters)
    optimizer.step()

    # Print status
    print(f'Epoch: {epoch + 1:02d} | Loss: {loss.item():.5f}')

## 11. Inspecting Learned Parameters
We inspect the weights and biases of our model's linear layer.

In [ ]:
# Inspect linear layer weights and biases (Fix: access through linear layer attribute)
print("Model Weights:\n", model.linear.weight)
print("Model Bias:\n", model.linear.bias)

## 12. Evaluating Model Accuracy on Test Data
We evaluate our model using classification predictions on the validation set, checking the accuracy.

In [ ]:
# Model Evaluation
with torch.no_grad():
    # Generate predictions on the validation test set
    test_preds = model(X_test_tensor)
    # Apply class decision threshold at 0.5
    binary_preds = (test_preds > 0.5).float()
    
    # Fix broadcasting issue by squeezing shape to match target vector y_test_tensor (critical bug fix)
    accuracy = (binary_preds.squeeze() == y_test_tensor).float().mean()
    
    print(f'Test Accuracy: {accuracy.item() * 100:.2f}%')